In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

In [3]:
set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini")

message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]

In [4]:
%%time
# 첫 번째 호출 - API 실제 호출
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

서울 광장시장에서 인기 있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤하고 달콤한 양념이 잘 어우러진 쌀떡으로, 광장시장에서 특히 인기가 많습니다.
2. **순대**: 돼지 창자에 찹쌀, 채소, 그리고 다양한 재료를 넣어 만든 순대로, 매콤한 양념장과 함께 먹으면 정말 맛있습니다.
3. **호떡**: 달콤한 시럽과 견과류가 들어간 부드러운 팬케이크로, 겨울철에 특히 인기가 많습니다.
4. **튀김**: 다양한 종류의 재료를 튀긴 음식으로, 특히 새우튀김이나 고구마튀김이 맛있습니다.
5. **상추쌈과 불고기**: 쌈장과 함께 제공되는 불고기도 아주 맛있습니다.

각 상점마다 특색이 있으니, 여러 가지를 시도해보는 것도 좋습니다!
CPU times: total: 78.1 ms
Wall time: 5.89 s


In [5]:
%%time
# 두 번째 호출 - 캐시에서 즉시 반환
response = llm.invoke(message)
print(response.content)
# Wall time: 약 1ms (거의 0)

서울 광장시장에서 인기 있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤하고 달콤한 양념이 잘 어우러진 쌀떡으로, 광장시장에서 특히 인기가 많습니다.
2. **순대**: 돼지 창자에 찹쌀, 채소, 그리고 다양한 재료를 넣어 만든 순대로, 매콤한 양념장과 함께 먹으면 정말 맛있습니다.
3. **호떡**: 달콤한 시럽과 견과류가 들어간 부드러운 팬케이크로, 겨울철에 특히 인기가 많습니다.
4. **튀김**: 다양한 종류의 재료를 튀긴 음식으로, 특히 새우튀김이나 고구마튀김이 맛있습니다.
5. **상추쌈과 불고기**: 쌈장과 함께 제공되는 불고기도 아주 맛있습니다.

각 상점마다 특색이 있으니, 여러 가지를 시도해보는 것도 좋습니다!
CPU times: total: 0 ns
Wall time: 0 ns


In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [7]:
from langchain_core.globals import set_llm_cache
from langchain_redis import RedisSemanticCache          # 변경
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage

In [3]:
REDIS_URL = os.environ['REDIS_URL']

In [4]:
semantic_cache = RedisSemanticCache(
    redis_url=REDIS_URL,
    embeddings=OpenAIEmbeddings(model="text-embedding-3-small"),  # 파라미터명 변경
    distance_threshold=0.2                                         # 파라미터명 변경
)

set_llm_cache(semantic_cache)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [6]:
print("RedisSemanticCache 연결 완료")

RedisSemanticCache 연결 완료


In [8]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")])
print(response.content)

코스피 지수는 한국 주식시장에 상장된 대기업들의 주가 동향을 종합하여 나타내는 지표입니다.
CPU times: total: 125 ms
Wall time: 2.07 s


In [10]:
%%time
response = llm.invoke([HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")]) # 동일 질문 재실행 (정확 일치 캐시)
print(response.content)

코스피 지수는 한국 주식시장에 상장된 대기업들의 주가 동향을 종합하여 나타내는 지표입니다.
CPU times: total: 15.6 ms
Wall time: 531 ms


In [11]:
%%time
response = llm.invoke([HumanMessage(content="코스피가 뭔지 간단히 설명해줘.")]) # 유사 질문
print(response.content)

코스피 지수는 한국 주식시장에 상장된 대기업들의 주가 동향을 종합하여 나타내는 지표입니다.
CPU times: total: 31.2 ms
Wall time: 333 ms
